# QTCAD M12 Builder — Result Analysis

Practical App Ge hole Part 1 (`1-builder_ge.py`) — KLayout 레이아웃(`.oas`, 다섯 게이트:
BL/BC/BR 배리어, P1/P2 플런저)을 `qtcad.builder.Builder`로 읽어 Ge/SiGe 헤테로구조를
압출(extrude)하고, dot 영역을 세분화한 뒤 3D 메시를 생성합니다.

**실행 메모**: `builder.view(..., save="*.svg")` 중간 시각화 호출들이 이 레이아웃에서
15분+/1100 CPU초 이상 끝나지 않아(메모리 문제는 아님, WorkingSet ~200MB 유지)
제거하고 메시 생성까지만 실행했습니다 — 핵심 산출물(메시)에는 영향 없습니다.
`.oas`/`.xao` 파일은 GDSTK/gmsh가 한글 경로를 못 열어 `C:\temp`를 경유했습니다.

**Base directory**

`C:\Users\norma\Desktop\연구자료\QTCAD Simulation\Simulation Example\M12. Builder\Reference\Ge_hole`

주요 산출물:
1. 메시 통계 (노드/요소 수)
2. 물리 그룹(게이트·재질·dot 영역) 확인
3. 최종 요약


In [1]:
# [동작] 기본 패키지 및 경로 설정
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path(
    r"C:\Users\norma\Desktop\연구자료\QTCAD Simulation\Simulation Example\M12. Builder\Reference\Ge_hole"
)
MESH_DIR = BASE_DIR / "meshes"
FILE_MSH = MESH_DIR / "ge_dqd.msh"
FILE_XAO = MESH_DIR / "ge_dqd.xao"

print("BASE_DIR:", BASE_DIR)
assert FILE_MSH.exists(), f"메시 파일이 없습니다:\n{FILE_MSH}"
print("[OK] 메시 파일 확인 완료")
print(f"ge_dqd.msh : {FILE_MSH.stat().st_size/1e6:.2f} MB")
print(f"ge_dqd.xao : {FILE_XAO.stat().st_size/1e3:.1f} KB" if FILE_XAO.exists() else "[주의] .xao 없음")


BASE_DIR: C:\Users\norma\Desktop\연구자료\QTCAD Simulation\Simulation Example\M12. Builder\Reference\Ge_hole
[OK] 메시 파일 확인 완료
ge_dqd.msh : 10.89 MB
ge_dqd.xao : 41.0 KB


## 1. 메시 통계

`.msh` 파일(Gmsh 포맷 4.1)의 헤더에서 노드/요소 수를 직접 파싱합니다.

In [2]:
# [점검] 메시 노드/요소 수 파싱
text = FILE_MSH.read_text(encoding="utf-8", errors="ignore")

m_nodes = re.search(r"\$Nodes\n(\d+) (\d+)", text)
m_elems = re.search(r"\$Elements\n(\d+) (\d+)", text)
n_entity_blocks_nodes, n_nodes = (int(x) for x in m_nodes.groups()) if m_nodes else (None, None)
n_entity_blocks_elems, n_elems = (int(x) for x in m_elems.groups()) if m_elems else (None, None)

stats_df = pd.DataFrame({
    "quantity": ["total nodes", "total elements", "entity blocks (nodes)", "entity blocks (elements)"],
    "value": [n_nodes, n_elems, n_entity_blocks_nodes, n_entity_blocks_elems],
})
display(stats_df)

print(
    "\n[해석] 이 메시는 손으로 쓴 .geo가 아니라 KLayout 마스크에서 자동 생성된 것이라, "
    "기준 소자(Si/SiGe, 01. SiGe One QD_part.1)의 손으로 만든 메시와 직접 node/E0/E1 "
    "비교를 할 대상이 없습니다 (재질도 Ge/SiGe 정공 소자로 다름). 이 노트북의 목적은 "
    "Builder 파이프라인(마스크 → 적층 압출 → dot 세분화 → 메시)이 실제로 끝까지 "
    "작동하는지 확인하는 것입니다. 기준 소자에 대한 node-count 비교를 하려면 우리 "
    "소자의 게이트 레이아웃을 .oas/.gds로 직접 만들어야 합니다 (이번 범위 밖)."
)


,quantity,value
0,total nodes,43102
1,total elements,279202
2,entity blocks (nodes),211
3,entity blocks (elements),55



[해석] 이 메시는 손으로 쓴 .geo가 아니라 KLayout 마스크에서 자동 생성된 것이라, 기준 소자(Si/SiGe, 01. SiGe One QD_part.1)의 손으로 만든 메시와 직접 node/E0/E1 비교를 할 대상이 없습니다 (재질도 Ge/SiGe 정공 소자로 다름). 이 노트북의 목적은 Builder 파이프라인(마스크 → 적층 압출 → dot 세분화 → 메시)이 실제로 끝까지 작동하는지 확인하는 것입니다. 기준 소자에 대한 node-count 비교를 하려면 우리 소자의 게이트 레이아웃을 .oas/.gds로 직접 만들어야 합니다 (이번 범위 밖).


## 2. 물리 그룹(게이트·재질·dot 영역) 확인

마스크의 폴리곤에 붙인 이름이 실제로 메시의 물리 그룹(게이트 표면, 재질 볼륨,
세분화된 dot 영역)으로 잘 전달됐는지 확인합니다.

In [3]:
# [점검] $PhysicalNames 블록 파싱
m_phys = re.search(r"\$PhysicalNames\n(\d+)\n(.*?)\n\$EndPhysicalNames", text, re.S)
n_phys = int(m_phys.group(1))
lines = m_phys.group(2).strip().split("\n")
rows = []
for line in lines:
    dim, tag, name = line.split(None, 2)
    rows.append({"dim": int(dim), "tag": int(tag), "name": name.strip('"')})
phys_df = pd.DataFrame(rows)
display(phys_df)

gates = sorted(phys_df.loc[(phys_df["dim"] == 2) & (phys_df["name"].str.len() <= 3), "name"].tolist())
volumes = phys_df.loc[phys_df["dim"] == 3, "name"].tolist()
print(f"\n검출된 게이트 표면 (짧은 이름): {gates}")
print(f"검출된 3D 재질/영역: {len(volumes)}개")

print(
    "\n[해석] 게이트 5개(BL/BC/BR 배리어, P1/P2 플런저)가 모두 물리 그룹으로 잡히고, "
    "substrate/SiGe_barrier/Ge_well/SiGe_cap/oxide 다섯 재질 층과 그 안의 "
    "'.dot_region' 세분화 볼륨이 함께 있으면, 마스크 → 적층 → dot 영역 세분화 "
    "파이프라인이 의도대로 작동한 것입니다."
)


,dim,tag,name
0,2,170,substrate_bottom
1,2,171,substrate_side
2,2,173,SiGe_barrier_bottom
3,2,174,SiGe_barrier_side
4,2,176,Ge_well_bottom
5,2,177,Ge_well_side
6,2,179,SiGe_cap_bottom
7,2,180,SiGe_cap_side
8,2,182,oxide_bottom
9,2,183,oxide_side



검출된 게이트 표면 (짧은 이름): ['BC', 'BL', 'BR', 'P1', 'P2']
검출된 3D 재질/영역: 8개

[해석] 게이트 5개(BL/BC/BR 배리어, P1/P2 플런저)가 모두 물리 그룹으로 잡히고, substrate/SiGe_barrier/Ge_well/SiGe_cap/oxide 다섯 재질 층과 그 안의 '.dot_region' 세분화 볼륨이 함께 있으면, 마스크 → 적층 → dot 영역 세분화 파이프라인이 의도대로 작동한 것입니다.


## 3. 최종 요약

In [4]:
# [점검] 핵심 결과 요약
print("=" * 88)
print("QTCAD M12 BUILDER — ANALYSIS SUMMARY")
print("=" * 88)
print(f"Total nodes              : {n_nodes:,}")
print(f"Total elements            : {n_elems:,}")
print(f"Physical groups           : {n_phys}")
print(f"Gate surfaces detected     : {gates}")
print(f"Material volumes detected  : {len(volumes)}")
print(f"Mesh file                  : {FILE_MSH}")
print("=" * 88)


QTCAD M12 BUILDER — ANALYSIS SUMMARY
Total nodes              : 43,102
Total elements            : 279,202
Physical groups           : 31
Gate surfaces detected     : ['BC', 'BL', 'BR', 'P1', 'P2']
Material volumes detected  : 8
Mesh file                  : C:\Users\norma\Desktop\연구자료\QTCAD Simulation\Simulation Example\M12. Builder\Reference\Ge_hole\meshes\ge_dqd.msh


## 해석 주의사항

- 중간 시각화(`builder.view(..., save=...)`)는 이 특정 레이아웃에서 비정상적으로 느려서
  (15분+, 메모리 문제 아님) 건너뛰었습니다 — 메시 자체의 정확성에는 영향이 없지만,
  마스크 → 메시 변환 과정을 눈으로 확인하는 중간 그림이 없다는 뜻입니다. 필요하면
  더 단순한 레이아웃으로 먼저 원인을 좁혀본 뒤 재시도해야 합니다.
- 이 메시는 `conf_carriers="h"`(정공) Ge/SiGe 소자용입니다. 우리 기준 소자(Si/SiGe
  전자)에 Builder를 적용하려면 같은 파이프라인을 우리 게이트 레이아웃(.oas/.gds)에
  대해 새로 실행해야 합니다.
- GaAs_gated 폴더의 "Gated Quantum Dot" 튜토리얼(`1-devicegen.py`)은 최신 `Builder`
  클래스가 아니라 구버전 `DeviceGenerator`(텍스트 레이아웃, 로컬 헬퍼 모듈 의존,
  대화형 `view()` 다수)를 쓰고 있어 이번 범위에서는 실행하지 않았습니다.
